In [1]:
import logging
import datetime
import numpy as np
from pathlib import Path
from keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from keras import Sequential, Input, Model
from keras.layers import Dense
from keras.models import load_model
from sktime.classification.deep_learning import (
    InceptionTimeClassifier,
    CNNClassifier,
    LSTMFCNClassifier,
    ResNetClassifier,
)
from sktime.classification.kernel_based import RocketClassifier
from sktime.classification.shapelet_based import ShapeletTransformClassifier
from sklearn.metrics import roc_auc_score, classification_report
from src.module.utils import load_npy

In [2]:
def balance_dataset(X, y):
    pos_idx, neg_idx = np.where(y == 1)[0], np.where(y == 0)[0]
    if len(pos_idx) > len(neg_idx):
        pos_idx = np.random.choice(pos_idx, size=len(neg_idx), replace=False)
    else:
        neg_idx = np.random.choice(neg_idx, size=len(pos_idx), replace=False)

    pos_X, pos_y = X[pos_idx], y[pos_idx]
    neg_X, neg_y = X[neg_idx], y[neg_idx]
    res_X = np.vstack([pos_X, neg_X])
    res_y = np.vstack([pos_y, neg_y])
    print(res_y.shape)
    logging.info(
        f"X: {res_X.shape} / y(pos/neg): {res_y.size}({sum(res_y==1)[0]}/{sum(res_y==0)[0]})"
    )
    return res_X, res_y


def init_callbacks(save_path) -> list:
    callbacks = [
        ModelCheckpoint(
            save_path,
            monitor="val_auc",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        ReduceLROnPlateau(
            monitor="val_auc", factor=0.1, patience=10, mode="max", verbose=1
        ),
        EarlyStopping(
            monitor="val_auc", patience=15, mode="max", start_from_epoch=15, verbose=1
        ),
    ]
    return callbacks


def replace_head(model, input_shape, n_classes) -> Model:
    keras_model = model.build_model(input_shape=input_shape, n_classes=n_classes)
    x = keras_model.layers[-2].output
    output = Dense(1, activation="sigmoid")(x)
    new_model = Model(inputs=keras_model.input, outputs=output)
    new_model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["auc"])
    return new_model


def train_model(model, train_X, train_y, val_X, val_y, callbacks):
    print(model.summary())
    history = model.fit(
        train_X,
        train_y,
        validation_data=(val_X, val_y),
        epochs=1000,
        batch_size=64,
        callbacks=callbacks,
        verbose=2,
    )
    return model, history


def evaluate_model(model, test_X, test_y, threshold):
    prob_y = model.predict(test_X)
    pred_y = np.where(prob_y >= threshold, 1, 0)
    print("AUROC: ", roc_auc_score(test_y, prob_y))
    print(classification_report(test_y, pred_y))

In [3]:
model_name = "resnet"
strategy = "hypophetversion2"
src_path = Path("../../data/06.train_val_test")

In [4]:
train_X = load_npy(src_path / f"vitaldb_{strategy}_train_X.npy")
train_y = load_npy(src_path / f"vitaldb_{strategy}_train_y.npy")
val_X = load_npy(src_path / f"vitaldb_{strategy}_val_X.npy")
val_y = load_npy(src_path / f"vitaldb_{strategy}_val_y.npy")
test_X = load_npy(src_path / f"vitaldb_{strategy}_test_X.npy")
test_y = load_npy(src_path / f"vitaldb_{strategy}_test_y.npy")

In [5]:
train_bal_X, train_bal_y = balance_dataset(train_X, train_y)

(43962, 1)


In [6]:
save_dir = Path("../../model")
save_dt = datetime.datetime.now().strftime("%y%m%d_%H%M%S")
save_path = save_dir / (save_dt + f"_{strategy}") / f"{model_name}.model.keras"

In [7]:
callbacks = init_callbacks(save_path)

In [ ]:
# clf = CNNClassifier(kernel_size=7, batch_size=32, n_conv_layers=2, random_state=42)
# clf = LSTMFCNClassifier(
#     dropout=0.8, kernel_sizes=(8, 5, 3), filter_sizes=(128, 256, 128), lstm_size=8
# )
# clf = ResNetClassifier(random_state=42)


In [10]:
clf = replace_head(clf, train_bal_X.shape[1:], 2)

2026-03-20 21:26:30.470265: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2 Max
2026-03-20 21:26:30.470301: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2026-03-20 21:26:30.470310: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 10.67 GB
2026-03-20 21:26:30.470329: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-20 21:26:30.470342: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [11]:
clf, history = train_model(clf, train_bal_X, train_bal_y, val_X, val_y, callbacks)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 3000, 1)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 3000, 64)  │        576 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 3000, 64)  │        256 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 3000, 64)  │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 3000, 64)  │     20,544 │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 3000, 64)  │        256 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 3000, 64)  │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 3000, 64)  │        128 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 3000, 64)  │     12,352 │ activation_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 3000, 64)  │        256 │ conv1d_3[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 3000, 64)  │        256 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 3000, 64)  │          0 │ batch_normalizat… │
│                     │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 3000, 64)  │          0 │ add[0][0]         │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 3000, 128) │     65,664 │ activation_2[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 3000, 128) │        512 │ conv1d_4[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 3000, 128) │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 3000, 128) │     82,048 │ activation_3[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 3000, 128) │        512 │ conv1d_5[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 3000, 128) │          0 │ batch_normalizat

 Total params: 506,689 (1.93 MB)

 Trainable params: 504,129 (1.92 MB)

 Non-trainable params: 2,560 (10.00 KB)

None
Epoch 1/1000


2026-03-20 21:26:35.721824: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.



Epoch 1: val_auc improved from None to 0.73700, saving model to ../../model/260320_212626_hypophetversion2/resnet.model.keras
687/687 - 147s - 214ms/step - auc: 0.7516 - loss: 0.5936 - val_auc: 0.7370 - val_loss: 0.6190 - learning_rate: 0.0010
Epoch 2/1000

Epoch 2: val_auc improved from 0.73700 to 0.74367, saving model to ../../model/260320_212626_hypophetversion2/resnet.model.keras
687/687 - 138s - 201ms/step - auc: 0.7679 - loss: 0.5767 - val_auc: 0.7437 - val_loss: 0.6064 - learning_rate: 0.0010
Epoch 3/1000

Epoch 3: val_auc improved from 0.74367 to 0.77415, saving model to ../../model/260320_212626_hypophetversion2/resnet.model.keras
687/687 - 138s - 201ms/step - auc: 0.7739 - loss: 0.5702 - val_auc: 0.7741 - val_loss: 0.6325 - learning_rate: 0.0010
Epoch 4/1000

Epoch 4: val_auc did not improve from 0.77415
687/687 - 138s - 200ms/step - auc: 0.7766 - loss: 0.5674 - val_auc: 0.6862 - val_loss: 0.7555 - learning_rate: 0.0010
Epoch 5/1000

Epoch 5: val_auc did not improve from 0.7

In [16]:
# load model
loaded = load_model(
    "../../model/260320_001228_hypophetversion2/inceptiontime.model.keras"
)

In [17]:
evaluate_model(loaded, test_X, test_y, 0.5)

468/468 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step
AUROC:  0.7818852573976414
              precision    recall  f1-score   support

         0.0       0.76      0.70      0.73      8052
         1.0       0.68      0.74      0.71      6914

    accuracy                           0.72     14966
   macro avg       0.72      0.72      0.72     14966
weighted avg       0.72      0.72      0.72     14966

